In [2]:
import csv
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import os
import glob

RANDOM_SEED = 42

# Dataset Path

In [14]:
model_save_path = 'model/keypoint_classifier/keypoint_sequence_classifier.keras'
tflite_save_path = 'model/keypoint_classifier/keypoint_sequence_classifier.tflite'

# Parameters

In [15]:
SEQUENCE_LENGTH = 25
FEATURES_PER_FRAME = 80  # 42 hand + 10 face + 8 pose + 20 relative

# Load Dataset

In [16]:
# Load Dataset
csv_files = sorted(glob.glob('Words-Dataset/*_sequence.csv'))
X_sequences = []
y_sequences = []
for csv_file in csv_files:
    data = np.loadtxt(csv_file, delimiter=',', dtype='float32')
    X_sequences.append(data[:, 1:].reshape(-1, SEQUENCE_LENGTH, FEATURES_PER_FRAME))
    y_sequences.extend(data[:, 0])

X_dataset = np.concatenate(X_sequences, axis=0)
y_dataset = np.array(y_sequences)
y_dataset -= 1  # Convert from 1-based to 0-based indexing for TensorFlow

X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, train_size=0.75, random_state=RANDOM_SEED)

# Load labels to determine number of classes
with open('Word-Label/keypoint_sequence_classifier_label.csv', encoding='utf-8-sig') as f:
    keypoint_sequence_classifier_labels = csv.reader(f)
    keypoint_sequence_classifier_labels = [row[0] for row in keypoint_sequence_classifier_labels]
NUM_CLASSES = len(keypoint_sequence_classifier_labels)
print(f"Number of classes: {NUM_CLASSES}")
print(f"Labels: {keypoint_sequence_classifier_labels}")

Number of classes: 100
Labels: ['你好', '學校', '同學', '屋企人', '鐘意', '唔鐘意', '點解', '彩虹', '謝謝', '等等', '對不起', '聾人', '我', '健聽', '\u2060手語', '現在', '高級', '認識/見面', '開心', '再見', '香港', '鑰匙', '護照', '護士', 'ok', '什麼', '麵', '紙巾', '有', '類別', '願望', '日本', '好', '懲罰', '講粗口', '人', '是', '不是', '需要', '不需要', '幫忙', '醫生', '咖啡', '想', '不想', '爸爸', '媽媽', '父母', '哥哥', '弟弟', '姐姐', '妹妹', '龍', '爺爺', '公公', '嫲嫲', '頭', '兒子', '女兒', '老公', '老婆', '頭痛', '頭盔', '我們', '鋼琴', '你', '戰爭', '句子', '學士', '衝突', '詞語', '工作', '標籤', '摩托車', '出糧', '噓', '厲害', '劍擊', '同事', '文職', '網絡', '運動', '學習', '緊張', '運動場', '腹瀉', '義工', '功課', '零', '一', '二', '三', '四', '睡覺', '六', '七', '八', '九', '助聽器', '盲']


# Build LSTM Model

In [17]:
if 'NUM_CLASSES' not in locals() or NUM_CLASSES == 0:
    print("No classes found. Please collect data first.")
else:
    from tensorflow.keras import layers
    model = tf.keras.models.Sequential([
        layers.Bidirectional(layers.LSTM(256, return_sequences=True), input_shape=(SEQUENCE_LENGTH, FEATURES_PER_FRAME)),
        layers.Dropout(0.3),
        layers.Bidirectional(layers.LSTM(128)),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional_4 (Bidirecti  (None, 25, 512)           690176    
 onal)                                                           
                                                                 
 dropout_6 (Dropout)         (None, 25, 512)           0         
                                                                 
 bidirectional_5 (Bidirecti  (None, 256)               656384    
 onal)                                                           
                                                                 
 dropout_7 (Dropout)         (None, 256)               0         
                                                                 
 dense_4 (Dense)             (None, 128)               32896     
                                                                 
 batch_normalization_2 (Bat  (None, 128)              

# Compile and Train Model

In [18]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

cp_callback = tf.keras.callbacks.ModelCheckpoint(model_save_path, verbose=1, save_weights_only=False)
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

model.fit(X_train, y_train, epochs=2000, batch_size=32, validation_data=(X_test, y_test), callbacks=[cp_callback, es_callback])

Epoch 1/2000
3467/3467 [==============================] - ETA: 0s - loss: 0.4125 - accuracy: 0.8926
Epoch 1: saving model to model/keypoint_classifier/keypoint_sequence_classifier.keras
3467/3467 [==============================] - 322s 92ms/step - loss: 0.4125 - accuracy: 0.8926 - val_loss: 0.0413 - val_accuracy: 0.9833
Epoch 2/2000
3467/3467 [==============================] - ETA: 0s - loss: 0.0790 - accuracy: 0.9764
Epoch 2: saving model to model/keypoint_classifier/keypoint_sequence_classifier.keras
3467/3467 [==============================] - 288s 83ms/step - loss: 0.0790 - accuracy: 0.9764 - val_loss: 0.0263 - val_accuracy: 0.9906
Epoch 3/2000
3467/3467 [==============================] - ETA: 0s - loss: 0.0551 - accuracy: 0.9837
Epoch 3: saving model to model/keypoint_classifier/keypoint_sequence_classifier.keras
3467/3467 [==============================] - 288s 83ms/step - loss: 0.0551 - accuracy: 0.9837 - val_loss: 0.0264 - val_accuracy: 0.9937
Epoch 4/2000
3467/3467 [==========

# Save a full SavedModel for OpenVINO conversion
Add a SavedModel export to ensure weights and graph are fully serialized.

In [ ]:
saved_model_dir = 'model/keypoint_classifier/keypoint_sequence_classifier_savedmodel'
if 'model' in locals():
    os.makedirs(saved_model_dir, exist_ok=True)
    model.save(saved_model_dir)
    print(f"SavedModel exported to: {saved_model_dir}")
else:
    print("Model not found. Train the model first.")

# Evaluate Model

In [19]:
val_loss, val_acc = model.evaluate(X_test, y_test)
print(f'Validation Loss: {val_loss}, Validation Accuracy: {val_acc}')

1156/1156 [==============================] - 24s 20ms/step - loss: 0.0052 - accuracy: 0.9986
Validation Loss: 0.0051566907204687595, Validation Accuracy: 0.9986207485198975


# Convert to TFLite

In [20]:
model.save(model_save_path)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

with open(tflite_save_path, 'wb') as f:
    f.write(tflite_model)

print("TFLite model saved.")

INFO:tensorflow:Assets written to: /var/folders/wt/bj47w0pj3h529wfvv7wn357w0000gn/T/tmpwgdntbfg/assets


INFO:tensorflow:Assets written to: /var/folders/wt/bj47w0pj3h529wfvv7wn357w0000gn/T/tmpwgdntbfg/assets


TFLite model saved.


2026-01-04 23:55:14.693566: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
2026-01-04 23:55:14.693764: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-01-04 23:55:14.695442: I tensorflow/cc/saved_model/reader.cc:45] Reading SavedModel from: /var/folders/wt/bj47w0pj3h529wfvv7wn357w0000gn/T/tmpwgdntbfg
2026-01-04 23:55:14.716933: I tensorflow/cc/saved_model/reader.cc:91] Reading meta graph with tags { serve }
2026-01-04 23:55:14.716943: I tensorflow/cc/saved_model/reader.cc:132] Reading SavedModel debug info (if present) from: /var/folders/wt/bj47w0pj3h529wfvv7wn357w0000gn/T/tmpwgdntbfg
2026-01-04 23:55:14.779778: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:375] MLIR V1 optimization pass is not enabled
2026-01-04 23:55:14.801266: I tensorflow/cc/saved_model/loader.cc:231] Restoring SavedModel bundle.
2026-01-04 23:55:14.976991: I tensorflow/cc/saved_model/loader.

# Test Inference

In [21]:
# Test Inference
interpreter = tf.lite.Interpreter(model_path=tflite_save_path)

# Add Flex delegate for TensorFlow ops
from tensorflow.lite.python.interpreter import load_delegate
try:
    delegate = load_delegate('libtensorflowlite_flex.so')
    interpreter = tf.lite.Interpreter(model_path=tflite_save_path, experimental_delegates=[delegate])
except:
    print("Flex delegate not available, using standard interpreter")

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

interpreter.set_tensor(input_details[0]['index'], np.array([X_test[0]], dtype=np.float32))
interpreter.invoke()
result = interpreter.get_tensor(output_details[0]['index'])

print("Predicted:", np.argmax(result))
print("Actual:", y_test[0])

INFO: Created TensorFlow Lite delegate for select TF ops.
INFO: TfLiteFlexDelegate delegate: 6 nodes delegated out of 34 nodes with 3 partitions.

Exception ignored in: <function Delegate.__del__ at 0x177552440>
Traceback (most recent call last):
  File "/Users/ronald8931/Desktop/doneeee/.venv/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py", line 109, in __del__
    if self._library is not None:
AttributeError: 'Delegate' object has no attribute '_library'


Flex delegate not available, using standard interpreter
Predicted: 34
Actual: 34.0


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
